Meshal Mahnoor

# Semantic Segmentation of Compost-Sorter Images with a U-Net

**Dataset:** Compost-Sorter (Roboflow Universe, `ecosorter/compost-sorter`, version 1), exported in **COCO Segmentation** format.
**Model:** U-Net trained from scratch.

**Why use U-Net, and why these settings?**
Mean IoU is a per-pixel (semantic) metric, so the task needs a label for every pixel, and the dataset is relatively small. A U-Net's skip connections preserve fine spatial detail, so it learns well from few images without a pretrained backbone. I use 128x128 images, batch size 32 and Adam (lr = 1e-3) so training is fast and fits in Colab GPU memory. I allow up to 30 epochs with early stopping (patience = 5) so training stops once the validation loss stops improving and does not overfit.

In [3]:
#roboflow dataset for compost segmentation:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="3uvMh6tX9dx05LfixTwv")
project = rf.workspace("ecosorter").project("compost-sorter")
version = project.version(1)
dataset = version.download("coco-segmentation")

loading Roboflow workspace...
loading Roboflow project...


## 1. Setup
Install `pycocotools` (to read COCO annotations), import the libraries and check the GPU.

In [4]:
!pip install -q pycocotools
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image
from pycocotools.coco import COCO
from tensorflow.keras import layers, Model

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
# All images and masks are resized to 128 x 128
IMG_SIZE = 128

TensorFlow: 2.21.0
GPUs: []


## 2. Locate the Dataset and Read the Classes
The Roboflow export has `train`, `valid` and `test` folders, each with its images and an `_annotations.coco.json` file. Class labels: **0 = background**, **1, 2, ... = dataset classes**.

In [5]:
# Find the folder that contains train/_annotations.coco.json
matches = glob.glob("/content/**/train/_annotations.coco.json", recursive=True)
if not matches:
    raise FileNotFoundError("Roboflow dataset not found under /content.")
DATA_DIR = os.path.dirname(os.path.dirname(matches[0]))
print("Dataset folder:", DATA_DIR)
print("Sub-folders   :", sorted(os.listdir(DATA_DIR)))

# Read the training annotations to discover the classes
coco_train = COCO(os.path.join(DATA_DIR, "train", "_annotations.coco.json"))

# Count how many annotations each category has
cat_counts = {}
for ann in coco_train.dataset["annotations"]:
    cat_counts[ann["category_id"]] = cat_counts.get(ann["category_id"], 0) + 1

# Keep only categories that are actually used and give each a label 1, 2, 3, ...
used_cat_ids = sorted(cat_counts.keys())
cat_to_label = {cid: i + 1 for i, cid in enumerate(used_cat_ids)}

# Class names, with the background at index 0
class_names = ["background"] + [coco_train.cats[cid]["name"] for cid in used_cat_ids]
NUM_CLASSES = len(class_names)

print("\nNumber of classes (including background):", NUM_CLASSES)
for i, n in enumerate(class_names):
    print(i, "->", n)

Dataset folder: /content/Compost-sorter-1
Sub-folders   : ['README.dataset.txt', 'README.roboflow.txt', 'test', 'train', 'valid']
loading annotations into memory...
Done (t=0.22s)
creating index...
index created!

Number of classes (including background): 6
0 -> background
1 -> 0 Compost
2 -> 1 Trash
3 -> 2 Recycle
4 -> 3 Person
5 -> object


## 3. Convert COCO Polygons into Segmentation Masks
A U-Net needs one label per pixel, whereas COCO stores objects as polygons. Each image is resized (bilinear) and its polygons are painted into one mask, resized with **nearest-neighbour** so labels stay whole numbers.

In [6]:
def load_split(split):
    # Read the COCO annotation file of this split
    coco = COCO(os.path.join(DATA_DIR, split, "_annotations.coco.json"))
    images, masks = [], []

    # Loop over every image of the split
    for img_id in sorted(coco.getImgIds()):
        info = coco.loadImgs(img_id)[0]
        h, w = info["height"], info["width"]

        # Load the RGB image and resize it
        img = Image.open(os.path.join(DATA_DIR, split, info["file_name"])).convert("RGB")
        img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)

        # Start with an all-background mask and paint every object into it
        mask = np.zeros((h, w), dtype=np.uint8)
        for ann in coco.loadAnns(coco.getAnnIds(imgIds=img_id, iscrowd=None)):
            if ann["category_id"] in cat_to_label:
                # annToMask converts the polygon to a binary mask
                m = coco.annToMask(ann)
                mask[m == 1] = cat_to_label[ann["category_id"]]

        # Resize the mask with nearest-neighbour so labels are not blended
        mask = Image.fromarray(mask).resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)

        images.append(np.array(img))
        masks.append(np.array(mask))

    # Images: (N, H, W, 3) uint8, masks: (N, H, W, 1) uint8
    return np.stack(images), np.stack(masks)[..., None]

train_x, train_y = load_split("train")
val_x, val_y = load_split("valid")
test_x, test_y = load_split("test")

print("Training examples  :", len(train_x))
print("Validation examples:", len(val_x))
print("Testing examples   :", len(test_x))
print("Image array shape  :", train_x.shape, " Mask array shape:", train_y.shape)

loading annotations into memory...
Done (t=0.06s)
creating index...
index created!
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Training examples  : 1944
Validation examples: 551
Testing examples   : 285
Image array shape  : (1944, 128, 128, 3)  Mask array shape: (1944, 128, 128, 1)


**Wrap the arrays as datasets.** Each sample is a dictionary with the keys `"image"` and `"segmentation_mask"`.

In [7]:
def to_dataset(x, y):
    return tf.data.Dataset.from_tensor_slices({
        "image": x,
        "segmentation_mask": y
    })

train_data = to_dataset(train_x, train_y)
val_data = to_dataset(val_x, val_y)
test_data = to_dataset(test_x, test_y)

## 4. Preprocessing and Input Pipelines
`preprocess()` resizes and normalises the image and mask together. Augmentation is used for the training set only.

In [8]:
# augment=True switches on random augmentation (used for the training set only)
def preprocess(sample, augment=False):

    # Resize the RGB image to IMG_SIZE x IMG_SIZE pixels
    image = tf.image.resize(
        # The original image from the dataset sample
        sample["image"],
        # Target size (height, width)
        (IMG_SIZE, IMG_SIZE)
    )

    # Resize the segmentation mask to the same size
    mask = tf.image.resize(
        # The original mask from the dataset sample
        sample["segmentation_mask"],
        # Target size (height, width)
        (IMG_SIZE, IMG_SIZE),
        # Nearest-neighbour keeps labels as whole numbers (no blended values)
        method="nearest"
    )

    # Convert pixels to float32 and scale from 0-255 to the range 0-1
    image = tf.cast(image, tf.float32) / 255.0

    # Convert labels to integers (labels are already 0 = background, 1.. = classes)
    mask = tf.cast(mask, tf.int32)

    # Apply augmentation only when it was requested
    if augment:

        # With 50% probability (random number above 0.5) ...
        if tf.random.uniform(()) > 0.5:
            # ... flip the image horizontally
            image = tf.image.flip_left_right(image)
            # ... and flip the mask in the same way so it still matches the image
            mask = tf.image.flip_left_right(mask)

        # Randomly change the brightness by up to +/- 0.1 (image only, never the mask)
        image = tf.image.random_brightness(image, 0.1)
        # Clip pixel values back into the valid range 0-1
        image = tf.clip_by_value(image, 0.0, 1.0)

    # Return the processed image and mask
    return image, mask

The training pipeline is augmented and shuffled; validation and test pipelines are only preprocessed and batched.

In [9]:
train_ds = (
    # Start from the training data
    train_data
    # Apply preprocess() to every sample with augmentation switched on
    .map(
        lambda x: preprocess(x, augment=True),
        # Process several samples in parallel; AUTOTUNE picks the number automatically
        num_parallel_calls=tf.data.AUTOTUNE
    )
    # Shuffle using a buffer of 1000 samples so each epoch sees a different order
    .shuffle(1000)
    # Group the samples into batches of 32 images
    .batch(32)
    # Prepare the next batch in the background while the GPU trains on the current one
    .prefetch(tf.data.AUTOTUNE)
)

# Validation pipeline (no augmentation, no shuffling)
val_ds = (
    val_data
    # Apply preprocess() with the default augment=False
    .map(
        preprocess,
        num_parallel_calls=tf.data.AUTOTUNE
    )
    # Group the samples into batches of 32 images
    .batch(32)
    # Prefetch batches in the background
    .prefetch(tf.data.AUTOTUNE)
)

# Test pipeline (no augmentation, no shuffling)
test_ds = (
    # Start from the test data
    test_data
    # Apply preprocess() with the default augment=False
    .map(
        preprocess,
        num_parallel_calls=tf.data.AUTOTUNE
    )
    # Group the samples into batches of 32 images
    .batch(32)
    # Prefetch batches in the background
    .prefetch(tf.data.AUTOTUNE)
)

## 5. Build the U-Net
Encoder (4 levels), bottleneck, decoder with skip connections. The last layer has one softmax output channel per class (background + dataset classes).

In [10]:
# conv_block(): two (Conv -> BatchNorm -> ReLU) layers, the basic unit of U-Net
def conv_block(x, filters):

    # 3x3 convolution with the given number of filters
    x = layers.Conv2D(
        filters,
        # Kernel size 3 (a 3x3 window)
        3,
        # "same" padding keeps the height and width unchanged
        padding="same",
        # he_normal is a weight initialization method used in NN and suits ReLU
        kernel_initializer="he_normal"
    )(x)

    # Normalise the activations to stabilise and speed up training
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # Second 3x3 convolution with the same settings
    x = layers.Conv2D(
        filters,
        3,
        padding="same",
        kernel_initializer="he_normal"
    )(x)

    # Second batch normalisation
    x = layers.BatchNormalization()(x)
    # Second ReLU activation
    x = layers.ReLU()(x)

    # Return the feature maps
    return x


# encoder_block(): extracts features and then halves the image size
def encoder_block(x, filters):

    # Features before pooling; they are kept as the skip connection for the decoder
    skip = conv_block(x, filters)
    # 2x2 max-pooling halves the height and width
    pooled = layers.MaxPooling2D(2)(skip)

    # Return the skip features and the pooled features
    return skip, pooled


# decoder_block(): doubles the image size and merges it with the encoder features
def decoder_block(x, skip, filters):

    # Transposed convolution (stride 2) up-samples the feature maps to twice the size
    x = layers.Conv2DTranspose(
        filters,
        # 2x2 kernel
        2,
        # Stride 2 doubles the height and width
        strides=2,
        padding="same"
    )(x)

    # Join the up-sampled maps with the matching encoder (skip) features
    x = layers.Concatenate()([x, skip])

    # Refine the combined features with a convolution block
    x = conv_block(x, filters)

    # Return the decoded feature maps
    return x


# build_unet(): assembles the complete network (default input is an IMG_SIZE x IMG_SIZE RGB image)
def build_unet(input_shape=(IMG_SIZE, IMG_SIZE, 3)):

    # Input layer that receives the image
    inputs = layers.Input(input_shape)

    # Encoder
    # Level 1: 32 filters, size / 2
    s1, p1 = encoder_block(inputs, 32)
    # Level 2: 64 filters, size / 4
    s2, p2 = encoder_block(p1, 64)
    # Level 3: 128 filters, size / 8
    s3, p3 = encoder_block(p2, 128)
    # Level 4: 256 filters, size / 16
    s4, p4 = encoder_block(p3, 256)

    # Bottleneck
    # Deepest layer of the network with 512 filters
    b = conv_block(p4, 512)

    # Decoder
    # Up-sample and join with skip s4
    d1 = decoder_block(b, s4, 256)
    # Up-sample and join with skip s3
    d2 = decoder_block(d1, s3, 128)
    # Up-sample and join with skip s2
    d3 = decoder_block(d2, s2, 64)
    # Up-sample to the original size and join with skip s1
    d4 = decoder_block(d3, s1, 32)

    # One output channel per class (background + dataset classes)
    outputs = layers.Conv2D(
        # One channel per class
        NUM_CLASSES,
        # 1x1 convolution works on each pixel separately
        1,
        # Softmax turns the values of every pixel into class probabilities
        activation="softmax"
    )(d4)

    # Create the Keras model from the input and output layers and name it "U-Net"
    return Model(inputs, outputs, name="U-Net")


model = build_unet()

## 6. Metric and Compilation
A custom Mean IoU metric that accepts softmax outputs, plus pixel accuracy, are tracked during training. The extra Dice metric is computed on the test set in section 9.

In [11]:
# Custom metric: inherits Keras' MeanIoU but accepts softmax probabilities as predictions
class MeanIoU(tf.keras.metrics.MeanIoU):

    # update_state() is called after every batch to update the running IoU
    def update_state(self, y_true, y_pred, sample_weight=None):

        # Convert the probabilities per pixel into the predicted class label
        y_pred = tf.argmax(y_pred, axis=-1)

        # Remove the extra channel dimension of the true mask
        y_true = tf.squeeze(y_true, axis=-1)

        # Pass the converted labels to the original MeanIoU implementation
        return super().update_state(
            y_true,
            y_pred,
            sample_weight
        )


model.compile(
    # Adam optimiser with a learning rate of 0.001
    optimizer=tf.keras.optimizers.Adam(1e-3),

    # Cross-entropy loss for integer labels and softmax probability outputs
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),

    metrics=[
        # Pixel accuracy: fraction of pixels classified correctly
        "accuracy",

        # Mean IoU over all classes, reported under the name "mean_iou"
        MeanIoU(num_classes=NUM_CLASSES, name="mean_iou")
    ]
)

## 7. Callbacks
Save the best model (by validation Mean IoU) and stop early after 5 epochs without improvement in validation loss.

In [12]:
callbacks = [

    # Save the model to disk whenever it improves
    tf.keras.callbacks.ModelCheckpoint(
        # File name of the saved model
        "best_unet.keras",
        # Watch the validation mean IoU
        monitor="val_mean_iou",
        # Higher mean IoU is better
        mode="max",
        save_best_only=True
    ),

    # Stop training when the model no longer improves
    tf.keras.callbacks.EarlyStopping(
        # Watch the validation loss
        monitor="val_loss",
        # Stop after 5 epochs without improvement
        patience=5,
        # Return to the weights of the best epoch when stopping
        restore_best_weights=True
    )
]

## 8. Train the Model
Up to 30 epochs; early stopping ends training sooner if validation loss stops improving.

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks
)

Epoch 1/30
39/61 ━━━━━━━━━━━━━━━━━━━━ 7:52 21s/step - accuracy: 0.6522 - loss: 1.2072 - mean_iou: 0.1833

**Observation:** Training stopped after epoch ___ because the validation loss had not improved for 5 epochs (best value ___ at epoch ___). The best validation Mean IoU was ___ at epoch ___, so that model was saved in `best_unet.keras`.

## 9. Evaluate on the Test Set
Reload the best checkpoint and evaluate on the test images (never used for training or validation): **pixel accuracy** and **Mean IoU**.

In [ ]:
# Load the best model saved by ModelCheckpoint
model = tf.keras.models.load_model(
    # File that was written during training
    "best_unet.keras",
    # Tell Keras how to rebuild our custom MeanIoU metric
    custom_objects={"MeanIoU": MeanIoU}
)

print("\nFinal evaluation on independent test set:")
test_results = model.evaluate(
    test_ds,
    return_dict=True
)

for metric, value in test_results.items():
    print(f"{metric}: {value:.4f}")

### Additional metric: Dice coefficient (F1 score)
Pixel accuracy can be misleading here because the background covers most pixels, so a model that predicts "background" everywhere would still score high.
Mean IoU already corrects for this. As the extra metric I chose the **Dice coefficient**, defined per class as `2·TP / (2·TP + FP + FN)`. It also measures the overlap between prediction and ground truth, but it is more forgiving than IoU for small objects, which suits compost items that occupy only a small part of the image.
Both metrics are computed per class from a confusion matrix over the whole test set, so we can also see which classes are hard for the model.

In [ ]:
# Collect true and predicted labels of every test pixel
y_true_all, y_pred_all = [], []

for images, masks in test_ds:
    # Predict a batch and convert probabilities to class labels
    preds = tf.argmax(model.predict(images, verbose=0), axis=-1)
    y_pred_all.append(preds.numpy().ravel())
    y_true_all.append(masks[..., 0].numpy().ravel())

y_true_all = np.concatenate(y_true_all)
y_pred_all = np.concatenate(y_pred_all)

# Confusion matrix: rows = true class, columns = predicted class
cm = tf.math.confusion_matrix(y_true_all, y_pred_all, num_classes=NUM_CLASSES).numpy().astype(np.float64)

# True positives, false positives and false negatives for every class
TP = np.diag(cm)
FP = cm.sum(axis=0) - TP
FN = cm.sum(axis=1) - TP

# Per-class IoU and Dice (classes absent from the test set give NaN and are ignored in the mean)
with np.errstate(divide="ignore", invalid="ignore"):
    iou = np.where(TP + FP + FN > 0, TP / (TP + FP + FN), np.nan)
    dice = np.where(2 * TP + FP + FN > 0, 2 * TP / (2 * TP + FP + FN), np.nan)

print(f"{'Class':<20}{'IoU':>8}{'Dice':>8}")
for i, n in enumerate(class_names):
    print(f"{n:<20}{iou[i]:>8.4f}{dice[i]:>8.4f}")

print("\nMean IoU  (all classes)       :", round(float(np.nanmean(iou)), 4))
print("Mean Dice (all classes)       :", round(float(np.nanmean(dice)), 4))
print("Mean IoU  (excluding background):", round(float(np.nanmean(iou[1:])), 4))
print("Mean Dice (excluding background):", round(float(np.nanmean(dice[1:])), 4))

**Observation:** On the unseen test set the model reached a pixel accuracy of ___, a Mean IoU of ___ and a mean Dice coefficient of ___. The foreground-only scores (___ IoU, ___ Dice) are lower/higher than the overall ones because the background is large and easy to predict. The best class was ___ and the weakest was ___.

## 10. Predictions Overlaid on Test Images
For each unseen test image: the original, the ground-truth mask overlay and the predicted mask overlay (background transparent, one colour per class).

In [ ]:
# Colours for the classes (background is not drawn)
palette = plt.get_cmap("tab10")(np.arange(NUM_CLASSES))[:, :3]

# overlay(): blends a coloured mask onto an image
def overlay(image, mask, alpha=0.5):
    # Copy the image so the original is not changed
    out = np.array(image).copy()
    # Pixels that belong to any class except the background
    fg = mask > 0
    # Blend the class colour with the image only on those pixels
    out[fg] = (1 - alpha) * out[fg] + alpha * palette[mask[fg]]
    return out


# show_overlays(): plots image, true overlay and predicted overlay for "num" test images
def show_overlays(dataset, num=5):
    # Take one batch from the dataset
    for images, masks in dataset.take(1):

        # Predict the masks for the whole batch
        preds = model.predict(images, verbose=0)

        # Loop over the first "num" images
        for i in range(min(num, len(images))):

            # Convert probabilities to one class label per pixel
            pred_mask = tf.argmax(preds[i], axis=-1).numpy()
            true_mask = masks[i][..., 0].numpy()

            plt.figure(figsize=(12, 4))
            for j, (item, title) in enumerate([
                (images[i].numpy(), "Image"),
                (overlay(images[i].numpy(), true_mask), "True mask overlay"),
                (overlay(images[i].numpy(), pred_mask), "Predicted mask overlay")
            ]):
                plt.subplot(1, 3, j + 1)
                plt.title(title)
                plt.imshow(item)
                plt.axis("off")
            plt.show()

    # Legend showing which colour belongs to which class
    plt.figure(figsize=(8, 0.6))
    for c in range(1, NUM_CLASSES):
        plt.scatter([], [], color=palette[c], label=class_names[c], s=80)
    plt.legend(ncol=NUM_CLASSES - 1, loc="center", frameon=False)
    plt.axis("off")
    plt.show()


# Show predictions for 5 images of the test set
show_overlays(test_ds, num=5)

**Observation:** The predicted masks follow the true objects ___ (closely / loosely). Typical errors are ___ (e.g. missed small objects, blurry boundaries, confusion between similar classes). Possible improvements: a larger image size, more augmentation, class-weighted loss, or a pretrained encoder.